# NC-03: Spatial Analysis

Loads the exported graph, converts it to NetworkX, then computes
standard space-syntax-inspired metrics: degree, closeness, and
betweenness centrality, shortest paths, and community detection.

**Run NC-02 first** to generate the CSV files.

In [1]:
from pathlib import Path

from topologicpy.Graph import Graph
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge

import networkx as nx
import pandas as pd

e:\softwares-4\graph-ml\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 0. Paths

In [2]:
BASE        = Path(r'E:\softwares-4\graph-ml\assign-04-node-classification')
GRAPHS_PATH = BASE / 'graphs'
print('Graphs folder:', GRAPHS_PATH)

Graphs folder: E:\softwares-4\graph-ml\assign-04-node-classification\graphs


## 1. Load graph from CSV

In [3]:
graphs = Graph.ByCSVPath(path=str(GRAPHS_PATH))
print('Graphs loaded:', len(graphs))

graph = graphs[0]
vertices = Graph.Vertices(graph) or []
edges    = Graph.Edges(graph)    or []
print('Vertices:', len(vertices))
print('Edges   :', len(edges))

# Show room types on each vertex
print()
print('Room type per node:')
for i, v in enumerate(vertices):
    d  = Topology.Dictionary(v)
    rt = Dictionary.ValueAtKey(d, 'room_type') or Dictionary.ValueAtKey(d, 'label') or '?'
    print(f'  node {i}: {rt}')

Graphs loaded: 1
Vertices: 10
Edges   : 13

Room type per node:
  node 0: 7
  node 1: 1
  node 2: ?
  node 3: 3
  node 4: 7
  node 5: 2
  node 6: 4
  node 7: ?
  node 8: ?
  node 9: 4


## 2. Convert to NetworkX

Builds an undirected NetworkX graph from the TopologicPy graph vertices
and edges, assigning room-type labels as node attributes.

In [4]:
G = nx.Graph()

# Build vertex index (TopologicPy vertex -> integer node id)
v_index = {}
for i, v in enumerate(vertices):
    d       = Topology.Dictionary(v)
    rt      = Dictionary.ValueAtKey(d, 'room_type') or ''
    label   = Dictionary.ValueAtKey(d, 'label')
    G.add_node(i, room_type=rt, label=label)
    v_index[i] = (round(Vertex.X(v), 2), round(Vertex.Y(v), 2), round(Vertex.Z(v), 2))

# Map coordinates back to index for edge lookup
coord_to_idx = {coord: idx for idx, coord in v_index.items()}

def vertex_id(v):
    key = (round(Vertex.X(v), 2), round(Vertex.Y(v), 2), round(Vertex.Z(v), 2))
    return coord_to_idx.get(key)

for e in edges:
    sv = Edge.StartVertex(e)
    ev = Edge.EndVertex(e)
    si = vertex_id(sv)
    ei = vertex_id(ev)
    if si is not None and ei is not None and si != ei:
        G.add_edge(si, ei)

print('NetworkX graph:')
print('  Nodes:', G.number_of_nodes())
print('  Edges:', G.number_of_edges())
print('  Connected:', nx.is_connected(G))

NetworkX graph:
  Nodes: 10
  Edges: 13
  Connected: True


## 3. Degree centrality

Measures how many rooms each room is directly connected to.

In [5]:
degree_centrality = nx.degree_centrality(G)

print('Degree centrality (normalised):')
for node, val in sorted(degree_centrality.items(), key=lambda x: -x[1]):
    rt = G.nodes[node].get('room_type', '?')
    print(f'  node {node:2d} ({rt:12s}): {val:.3f}  [degree={G.degree(node)}]')

Degree centrality (normalised):
  node  0 (            ): 0.333  [degree=3]
  node  1 (            ): 0.333  [degree=3]
  node  3 (            ): 0.333  [degree=3]
  node  4 (            ): 0.333  [degree=3]
  node  6 (            ): 0.333  [degree=3]
  node  9 (            ): 0.333  [degree=3]
  node  2 (            ): 0.222  [degree=2]
  node  5 (            ): 0.222  [degree=2]
  node  7 (            ): 0.222  [degree=2]
  node  8 (            ): 0.222  [degree=2]


## 4. Closeness centrality

How quickly a room can reach all other rooms (integration / access efficiency).

In [6]:
closeness_centrality = nx.closeness_centrality(G)

print('Closeness centrality:')
for node, val in sorted(closeness_centrality.items(), key=lambda x: -x[1]):
    rt = G.nodes[node].get('room_type', '?')
    print(f'  node {node:2d} ({rt:12s}): {val:.3f}')

Closeness centrality:
  node  1 (            ): 0.529
  node  3 (            ): 0.529
  node  6 (            ): 0.429
  node  9 (            ): 0.429
  node  2 (            ): 0.409
  node  5 (            ): 0.409
  node  0 (            ): 0.346
  node  4 (            ): 0.346
  node  7 (            ): 0.333
  node  8 (            ): 0.333


## 5. Betweenness centrality

How often a room lies on the shortest path between other rooms (choice / movement potential).

In [7]:
betweenness_centrality = nx.betweenness_centrality(G, normalized=True)

print('Betweenness centrality:')
for node, val in sorted(betweenness_centrality.items(), key=lambda x: -x[1]):
    rt = G.nodes[node].get('room_type', '?')
    print(f'  node {node:2d} ({rt:12s}): {val:.3f}')

Betweenness centrality:
  node  1 (            ): 0.569
  node  3 (            ): 0.569
  node  6 (            ): 0.250
  node  9 (            ): 0.250
  node  2 (            ): 0.083
  node  5 (            ): 0.083
  node  0 (            ): 0.042
  node  4 (            ): 0.042
  node  7 (            ): 0.000
  node  8 (            ): 0.000


## 6. Shortest paths

Pair-wise shortest path lengths between all rooms.

In [8]:
spl = dict(nx.all_pairs_shortest_path_length(G))

print('Shortest path lengths (node pairs):')
for src in sorted(spl.keys()):
    src_rt = G.nodes[src].get('room_type', '?')
    for dst in sorted(spl[src].keys()):
        if dst <= src:
            continue
        dst_rt = G.nodes[dst].get('room_type', '?')
        d = spl[src][dst]
        print(f'  {src_rt:12s} -> {dst_rt:12s}: {d} hop(s)')

Shortest path lengths (node pairs):
               ->             : 3 hop(s)
               ->             : 1 hop(s)
               ->             : 2 hop(s)
               ->             : 5 hop(s)
               ->             : 4 hop(s)
               ->             : 1 hop(s)
               ->             : 5 hop(s)
               ->             : 1 hop(s)
               ->             : 4 hop(s)
               ->             : 2 hop(s)
               ->             : 1 hop(s)
               ->             : 2 hop(s)
               ->             : 1 hop(s)
               ->             : 2 hop(s)
               ->             : 2 hop(s)
               ->             : 3 hop(s)
               ->             : 1 hop(s)
               ->             : 1 hop(s)
               ->             : 4 hop(s)
               ->             : 3 hop(s)
               ->             : 2 hop(s)
               ->             : 4 hop(s)
               ->             : 2 hop(s)
               ->    

## 7. Community detection

Grouping rooms into spatial communities using the Louvain method.

In [9]:
try:
    from networkx.algorithms.community import louvain_communities
    communities = louvain_communities(G, seed=42)
    print(f'Communities found: {len(communities)}')
    for i, comm in enumerate(communities):
        room_names = [G.nodes[n].get('room_type', '?') for n in sorted(comm)]
        print(f'  Community {i}: {room_names}')
except ImportError:
    from networkx.algorithms.community import greedy_modularity_communities
    communities = list(greedy_modularity_communities(G))
    print(f'Communities found: {len(communities)}')
    for i, comm in enumerate(communities):
        room_names = [G.nodes[n].get('room_type', '?') for n in sorted(comm)]
        print(f'  Community {i}: {room_names}')

Communities found: 2
  Community 0: ['', '', '', '', '']
  Community 1: ['', '', '', '', '']


## 8. Summary table

In [10]:
rows = []
for node in sorted(G.nodes()):
    rows.append({
        'node_id'             : node,
        'room_type'           : G.nodes[node].get('room_type', '?'),
        'label'               : G.nodes[node].get('label', -1),
        'degree'              : G.degree(node),
        'degree_centrality'   : round(degree_centrality[node], 4),
        'closeness_centrality': round(closeness_centrality[node], 4),
        'betweenness_centrality': round(betweenness_centrality[node], 4),
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

# Save to CSV
out_csv = GRAPHS_PATH / 'spatial_analysis.csv'
summary_df.to_csv(out_csv, index=False)
print()
print('Saved to:', out_csv)

 node_id room_type  label  degree  degree_centrality  closeness_centrality  betweenness_centrality
       0                7       3             0.3333                0.3462                  0.0417
       1                1       3             0.3333                0.5294                  0.5694
       2                0       2             0.2222                0.4091                  0.0833
       3                3       3             0.3333                0.5294                  0.5694
       4                7       3             0.3333                0.3462                  0.0417
       5                2       2             0.2222                0.4091                  0.0833
       6                4       3             0.3333                0.4286                  0.2500
       7                0       2             0.2222                0.3333                  0.0000
       8                0       2             0.2222                0.3333                  0.0000
       9  